In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_syndicated_facility_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "SyndicatedFacilityId String"
        # ",SourceSystem String"
        # ",PortfolioKey String"
        # ",PortfolioName String"
        # ",IngestionTimestamp  timestamp"
        # ",BatchId	String"
        # ",SyndicatedFacilityKey String"
        # ",FacilityKey String"
        # ",AgreementNumber String"
        # ",ExposureKey String"
        # ",LenderPartyKey String"
        # ",LenderPartyTaxIdentificationNumber String"
        # ",IsLeaderFlag boolean"
        # ",PartyPercentage decimal(19,2)"
        # ",EntryStatus string"
        # ",CapitalInitialBalanceAmount decimal(19,2)"
        # ",CapitalCurrentBalanceAmount decimal(19,2)"
        # ",CapitalTotalDrawnAmount decimal(19,2)"
        # ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.SyndicatedFacility"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "profits",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== SyndicatedFacility_profits ==================

        SyndicatedFacility_extra_filters = None

        if SyndicatedFacility_extra_filters is not None:
            SyndicatedFacility_extra_filters.append(date_filter)
            SyndicatedFacility_final_filters = " AND ".join(
                SyndicatedFacility_extra_filters
            )
        else:
            SyndicatedFacility_final_filters = date_filter

            SyndicatedFacility_extra_filters = None

        SyndicatedFacility_selected_fields = [
            "FACILITY_ID",
            "CUSTOMER_ID",
            "LNS_ACCOUNT_NUMBER",
            "CAPITAL_TOTAL_DRAWN",
            "ENTRY_STATUS",
            "SYNDICATED_AGREEMENT_NUM",
            "businessdate",
        ]

        SyndicatedFacility_profits = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_syndicatedfacility",  # tobechecked
            SyndicatedFacility_final_filters,
            SyndicatedFacility_selected_fields,
        )

        # Declaring a second table that will be used to join with SyndicatedParticipants but in both FACILITY_ID and CUSTOMER_ID columns

        SyndicatedFacility_profits_FacilityId = SyndicatedFacility_profits.alias(
            "SyndicatedFacility_profits_FacilityId"
        )

        SyndicatedFacility_profits_FacilityId_CustomerId = (
            SyndicatedFacility_profits.alias(
                "SyndicatedFacility_profits_FacilityId_CustomerId"
            )
        )

        SyndicatedFacility_profits_FacilityId_CustomerId = (
            SyndicatedFacility_profits_FacilityId_CustomerId.withColumnRenamed(
                "CUSTOMER_ID", "CUSTOMER_ID_FC"
            )
            .withColumnRenamed("FACILITY_ID", "FACILITY_ID_FC")
            .withColumnRenamed("CAPITAL_TOTAL_DRAWN", "CAPITAL_TOTAL_DRAWN_FC")
        )

        # ================== SyndicatedParticipants ==================

        SyndicatedParticipants_extra_filters = None

        if SyndicatedParticipants_extra_filters is not None:
            SyndicatedParticipants_extra_filters.append(date_filter)
            SyndicatedParticipants_final_filters = " AND ".join(
                SyndicatedParticipants_extra_filters
            )
        else:
            SyndicatedParticipants_final_filters = date_filter

        SyndicatedParticipants_selected_fields = [
            "FACILITY_ID",
            "CUSTOMER_ID",
            "IS_LEADER",
            "CUSTOMER_PERCENTAGE",
            "CAPITAL_INITIAL_BAL",
            "CAPITAL_CURRENT_BAL",
            "businessdate",
        ]

        SyndicatedParticipants_profits = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_syndicatedparticipants",
            SyndicatedParticipants_final_filters,
            SyndicatedParticipants_selected_fields,
        )

        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "cust_id",
            "cdi",
            "unit_code",
            "TAX_NUMBER",
            "businessdate",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_customer",
            customer_final_filters,
            customer_selected_fields,
        )

        customer_lender = customer.alias("customer_lender")

        # ==============================SyndicatedFacility_final====================

        # Conditions
        SyndicatedFacility_SyndicatedParticipants_join_condition_FacilityId = (
            SyndicatedFacility_profits_FacilityId["FACILITY_ID"]
            == SyndicatedParticipants_profits["FACILITY_ID"]
        )

        SyndicatedFacility_SyndicatedParticipants_join_condition_FacilityId_CustomerId = (
            SyndicatedFacility_profits_FacilityId_CustomerId["FACILITY_ID_FC"]
            == SyndicatedParticipants_profits["FACILITY_ID"]
        ) & (
            SyndicatedFacility_profits_FacilityId_CustomerId["CUSTOMER_ID_FC"]
            == SyndicatedParticipants_profits["CUSTOMER_ID"]
        )

        SyndicatedFacility_portfolio_join_condition = (
            substring(SyndicatedFacility_profits_FacilityId["LNS_ACCOUNT_NUMBER"], 1, 3)
            == portfolio["Prefix"]
        )

        SyndicatedFacility_customer_join_condition_lender = (
            SyndicatedParticipants_profits["CUSTOMER_ID"] == customer_lender["CUST_ID"]
        )

        # Join
        SyndicatedFacility_target_table = (
            SyndicatedParticipants_profits.join(
                SyndicatedFacility_profits_FacilityId,
                SyndicatedFacility_SyndicatedParticipants_join_condition_FacilityId,
                "inner",
            )
            .join(
                SyndicatedFacility_profits_FacilityId_CustomerId,
                SyndicatedFacility_SyndicatedParticipants_join_condition_FacilityId_CustomerId,
                "left",
            )
            .join(
                customer_lender,
                SyndicatedFacility_customer_join_condition_lender,
                "left",
            )
            .join(
                portfolio,
                SyndicatedFacility_portfolio_join_condition,
                "left",
            )
        )

        # ================ SyndicatedFacility_profits ============

        SyndicatedFacility_target_table_primary_key_fields = [
            SyndicatedParticipants_profits.FACILITY_ID,
            SyndicatedParticipants_profits.CUSTOMER_ID,
            lit("profits"),
        ]

        SyndicatedFacility_key_fields = [
            SyndicatedParticipants_profits.FACILITY_ID,
            SyndicatedParticipants_profits.CUSTOMER_ID,
        ]

        SyndicatedFacility_target_table = SyndicatedFacility_target_table.withColumn(
            "SyndicatedFacilityID",
            sha2(
                concat_ws(";", *SyndicatedFacility_target_table_primary_key_fields),
                256,
            ),
        ).withColumn(
            "SyndicatedFacilityKey",
            concat_ws(";", *SyndicatedFacility_key_fields),
        )

        SyndicatedFacility = SyndicatedFacility_target_table.select(
            SyndicatedFacility_target_table.SyndicatedFacilityID.alias(
                "SyndicatedFacilityID"
            ),
            lit("profits").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            SyndicatedFacility_target_table.SyndicatedFacilityKey.cast("string").alias(
                "SyndicatedFacilityKey"
            ),
            SyndicatedFacility_profits_FacilityId.FACILITY_ID.cast("string").alias(
                "FacilityKey"
            ),
            SyndicatedFacility_profits_FacilityId.SYNDICATED_AGREEMENT_NUM.cast(
                "string"
            ).alias("AgreementNumber"),
            SyndicatedFacility_profits_FacilityId.LNS_ACCOUNT_NUMBER.cast(
                "string"
            ).alias("ExposureKey"),
            customer_lender.cust_id.cast("string").alias("LenderPartyKey"),
            customer_lender.TAX_NUMBER.cast("string").alias(
                "LenderPartyTaxIdentificationNumber"
            ),
            SyndicatedParticipants_profits.IS_LEADER.cast("boolean").alias(
                "IsLeaderFlag"
            ),
            SyndicatedParticipants_profits.CUSTOMER_PERCENTAGE.cast(
                "decimal(19,2)"
            ).alias("PartyPercentage"),
            SyndicatedFacility_profits_FacilityId.ENTRY_STATUS.cast("string").alias(
                "EntryStatus"
            ),
            SyndicatedParticipants_profits.CAPITAL_INITIAL_BAL.cast(
                "decimal(19,2)"
            ).alias("CapitalInitialBalanceAmount"),
            SyndicatedParticipants_profits.CAPITAL_CURRENT_BAL.cast(
                "decimal(19,2)"
            ).alias("CapitalCurrentBalanceAmount"),
            SyndicatedFacility_profits_FacilityId_CustomerId.CAPITAL_TOTAL_DRAWN_FC.cast(
                "decimal(19,2)"
            ).alias(
                "CapitalTotalDrawnAmount"
            ),
            customer.businessdate.alias("BusinessDate"),
        )
        # ================== Append ==================

        SyndicatedFacility.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_syndicated_facility_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)